# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rufatj/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is fundamentally a **ranking / scoring** task ("which ones first?"), built on top of a
**classification** sub-task. Concretely: I first train a classifier that estimates the probability
a page belongs to the label of interest (section 2), then turn that probability into a ranked
score, blended with a transparent rule-based score -- exactly like the starter pipeline's
`final_refresh_score = 100 * (0.70 * best_model_probability + 0.30 * normalized_baseline_score)`
(`docs/ml-intern-dataset-and-lane-guide.md`). The classifier alone is not the deliverable; the
ranked, reason-coded queue it produces is.


In [1]:
import os

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "run this from inside the repo"

import pandas as pd

# Check: the task-type mapping this choice follows (framing-ml-problems skill).
mapping = {
    "Which ones first?": "Ranking / scoring -> precision@K",
    "Will this one decline / recover?": "Classification -> ROC-AUC, precision/recall vs base rate",
}
for q, t in mapping.items():
    print(f"{q!r:35} -> {t}")
print("\nThis lane is primarily the first row, using a model like the second row as its scoring engine.")


'Which ones first?'                 -> Ranking / scoring -> precision@K
'Will this one decline / recover?'  -> Classification -> ROC-AUC, precision/recall vs base rate

This lane is primarily the first row, using a model like the second row as its scoring engine.


## 2. Target or proxy

**Starter proxy (what ships today):** `is_declining_label = (trend_direction == "down")`. This is
a **defined rule**, not a directly observed future outcome -- `trend_direction` itself is computed
from `trend_pct`, which compares the last 30 days to the previous 30 days inside the *same* export
window. I'm using it for now because it's the only label the small starter CSV can support (a
single 90-day snapshot has no real forward-looking window), and the starter results in
`outputs/model_report.md` are honestly reported as coming from this proxy, on a 30k-row slice.

**Where I'm taking it (weeks 3+, full warehouse):** a genuinely **observed future outcome** --
`features from the prior 90 days -> decline (or recovery) over the next 30 days`, built from
`fact_content_daily_performance`'s ~17 months of daily history. That removes the proxy's biggest
weakness: right now "declining" is a same-window bucket, not something that actually happened
afterward. `trend_direction` and `trend_pct` stay excluded as model features either way, since they
define the label itself -- the leakage trap flagged in `docs/data-dictionary.md`.


In [2]:
import re

# Check: confirm the label-defining columns are genuinely excluded from the feature lists,
# not just excluded "in theory".
src = open("scripts/ml_utils.py").read()
for name in ["trend_direction", "trend_pct"]:
    for feat_list in ["MODEL_NUMERIC_FEATURES", "MODEL_CATEGORICAL_FEATURES"]:
        block = re.search(rf"{feat_list}\s*=\s*\[(.*?)\]", src, re.S)
        in_list = bool(block) and name in block.group(1)
        print(f"{name!r:18} in {feat_list:28}: {in_list}")


'trend_direction'  in MODEL_NUMERIC_FEATURES      : False
'trend_direction'  in MODEL_CATEGORICAL_FEATURES  : False
'trend_pct'        in MODEL_NUMERIC_FEATURES      : False
'trend_pct'        in MODEL_CATEGORICAL_FEATURES  : False


## 3. Success metric

**Metric: Precision@50** -- of the top 50 pages the ranked queue says to review first, how many
are actually positive by the label above? I'm choosing this over ROC-AUC or plain accuracy because
reviewer capacity, not overall correctness, is the real constraint (section 2 of
`w01_research_question.ipynb`): nobody works through all 30,000 (or 519,606, at warehouse scale)
pages -- they work through the top of the list. On the committed starter run, the hand-written rule
reaches Precision@50 = 0.240 (12 of the top 50 right) and the random forest reaches 0.740 (37 of
the top 50 right) -- `outputs/model_report.md`. "Good" for my own capstone means beating the rule
baseline on this same metric, on a validation split that holds out whole clients (`client_holdout`),
not just rows.


In [3]:
report = open("outputs/model_report.md").read()
for line in report.splitlines():
    if line.startswith("| random_forest") or line.startswith("| baseline_rules"):
        print(line)
print(
    "\nTarget for my own capstone: Precision@50 measurably above the baseline row above, on a "
    "client-holdout split, on the full-warehouse label (prior 90d -> next 30d)."
)


| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

Target for my own capstone: Precision@50 measurably above the baseline row above, on a client-holdout split, on the full-warehouse label (prior 90d -> next 30d).


## 4. The unit of analysis, as a real dataframe

**One row = one content item (page)**, aggregated over a trailing 90-day window -- not one visit,
one query, or one day. That is the grain of `data/raw/content_refresh_anonymized.csv` today, and it
stays the grain once I move to the warehouse's `dim_content` + `fact_content_daily_performance`
(joined and re-aggregated back onto one-row-per-page-per-decision-point, keyed by `content_hash_id`).


In [4]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(
    f"Rows (pages): {len(df):,}  |  Columns: {df.shape[1]}  |  Distinct clients: "
    f"{df['client_id'].nunique()}"
)

grain_check = df.groupby("content_id").size()
print(f"Max rows per content_id: {grain_check.max()} -> confirms one row per page, no duplicates.")

df[["content_id", "client_id", "content_type", "impressions_90d", "avg_position", "trend_direction"]].head(5)


Rows (pages): 30,000  |  Columns: 44  |  Distinct clients: 32
Max rows per content_id: 1 -> confirms one row per page, no duplicates.


,content_id,client_id,content_type,impressions_90d,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,down


## 5. Why ML beats a fixed rule here

Because the signals that matter interact, and the interactions are non-linear: a stale page only
matters if it still has demand; a low CTR only matters relative to its own position tier, not in
absolute terms; a declining trend only matters combined with volume. `outputs/model_report.md`
makes this concrete on the starter slice: the hand-written rule baseline (a fixed set of
if-thresholds) reaches ROC AUC 0.627, while a random forest trained on the same rows and the same
label reaches 0.750, and roughly triples Precision@50 (0.240 -> 0.740). That gap is what "too messy
for an if-statement" looks like in numbers -- a real, learnable pattern, not one a single fixed rule
already captures. A plain rule stays genuinely useful here too -- as the transparent, always-on
baseline the model has to beat, and as the source of the reason codes reviewers see next to each
page's score.


In [5]:
report = open("outputs/model_report.md").read()
start = report.index("## Top Features")
end = report.index("## Top 10 Queue Preview")
print(report[start:end].strip())


## Top Features

- `days_with_impressions`: 0.1578
- `log_impressions_90d`: 0.1282
- `avg_position`: 0.1090
- `content_age_days`: 0.0955
- `char_count`: 0.0426
- `word_count`: 0.0397
- `log_clicks_90d`: 0.0346
- `ctr`: 0.0330
- `scroll_rate`: 0.0311
- `days_with_sessions`: 0.0280


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.
